# Efficient LR-QAOA vs WalkSAT benchmark (BM24)

Rigorous comparison aligned with `bm24_qaoa_sim.py` / `train_lr_notebook_protocol.py`.

**Unlike `sweep_lr_depth_until_win.py`, this notebook always runs every depth in `CFG["depths"]` — it does not stop early when LR beats WalkSAT.**

1. **SAT benchmark set built once** — `H_diag` and Numba clause arrays cached per instance.
2. **WalkSAT + WalkSATlm run once** (Numba) — not repeated every QAOA depth.
3. **Training set at `n_train` built once** — reused for every depth.
4. **Per depth:** train `(dg, db)` + LR-QAOA benchmark only (BM24 `run_qaoa`).

**Y-axis (summary plot):** log₂ slope of **median(1/p_succ)** vs `n` (shot-cost exponent; lower = milder growth with `n`).

**Classical parameters in `CFG`:**
- `walksat_p_noise` — random-walk probability for WalkSAT (**0.5**).
- `walksatlm_p_noise` — random-walk probability for WalkSATlm (**0.15**).
- `walksatlm_w1`, `walksatlm_w2` — WalkSATlm linear-make weights (default 6 and 5).

In [1]:
# --- Configuration (edit here) ---
from pathlib import Path

CFG = {
    "k": 8,
    "r": 176.54,
    "seed": 27,
    "train_n": 12,
    "train_size": 50,
    "n_min": 12,
    "n_max": 20,
    "test_size": 200,
    "depths": list(range(2, 30)),  # or e.g. [2, 3, 4, 5, 10, 15]
    "skip_grid": False,           # True = notebook deep-sweep style (COBYLA only)
    "cobyla_maxiter": 200,
    "walksat_p_noise": 0.5,
    "walksatlm_p_noise": 0.15,
    "walksatlm_w1": 6,            # WalkSATlm only: weight on make-1 clauses when tie-breaking
    "walksatlm_w2": 5,            # WalkSATlm only: weight on make-2 clauses
    "annotate_first_win": False,  # plot only; never stops the depth loop early
    "max_flips": 100_000,
    "lr_beta_schedule": "decreasing",
    "output_dir": Path("bm24_runs"),
}
CFG["output_dir"].mkdir(parents=True, exist_ok=True)
print(CFG)

{'k': 8, 'r': 176.54, 'seed': 27, 'train_n': 12, 'train_size': 50, 'n_min': 12, 'n_max': 20, 'test_size': 200, 'depths': [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29], 'skip_grid': False, 'cobyla_maxiter': 200, 'p_noise': 0.5, 'walksatlm_w1': 6, 'walksatlm_w2': 5, 'annotate_first_win': False, 'max_flips': 100000, 'lr_beta_schedule': 'decreasing', 'output_dir': PosixPath('bm24_runs')}


In [2]:
import json
import sys
import time
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
from numba import njit
from scipy.stats import linregress
from tqdm.auto import tqdm

_PHASECRAFT = Path.cwd() if (Path.cwd() / "bm24_qaoa_sim.py").is_file() else Path.cwd() / "phasecraft"
if str(_PHASECRAFT) not in sys.path:
    sys.path.insert(0, str(_PHASECRAFT))

from bm24_qaoa_sim import (
    build_h_diagonal,
    generate_random_clause,
    generate_random_formula,
    make_lr_angles,
    per_instance_success_probability,
    run_qaoa,
)
from train_lr_notebook_protocol import (
    generate_training_h_diagonals,
    train_lr_grid_search_bm24,
)

K = int(CFG["k"])
LN2 = float(np.log(2.0))

In [3]:
# --- Numba classical solvers (notebook-accelerated; k literals per clause) ---

@njit(cache=True)
def fast_walksat_solver(n, c_vars, c_signs, max_flips, p_noise):
    assignment = np.random.randint(0, 2, n)
    m, k_sat = c_vars.shape
    unsat_buffer = np.empty(m, dtype=np.int32)
    for flip in range(max_flips):
        unsat_count = 0
        for i in range(m):
            is_sat = False
            for kk in range(k_sat):
                v = c_vars[i, kk]
                if assignment[v] == c_signs[i, kk]:
                    is_sat = True
                    break
            if not is_sat:
                unsat_buffer[unsat_count] = i
                unsat_count += 1
        if unsat_count == 0:
            return flip + 1
        target_c_idx = unsat_buffer[np.random.randint(0, unsat_count)]
        if np.random.random() < p_noise:
            var_to_flip = c_vars[target_c_idx, np.random.randint(0, k_sat)]
        else:
            best_var = -1
            min_breaks = 999999
            for kk in range(k_sat):
                candidate_var = c_vars[target_c_idx, kk]
                assignment[candidate_var] = 1 - assignment[candidate_var]
                current_breaks = 0
                for i_scan in range(m):
                    c_sat = False
                    for kk2 in range(k_sat):
                        v_scan = c_vars[i_scan, kk2]
                        if assignment[v_scan] == c_signs[i_scan, kk2]:
                            c_sat = True
                            break
                    if not c_sat:
                        current_breaks += 1
                if current_breaks < min_breaks:
                    min_breaks = current_breaks
                    best_var = candidate_var
                assignment[candidate_var] = 1 - assignment[candidate_var]
            var_to_flip = best_var
        assignment[var_to_flip] = 1 - assignment[var_to_flip]
    return max_flips


@njit(cache=True)
def walksatlm_paper_kernel(n, c_vars, c_signs, max_flips, p_noise, w1, w2):
    m = c_vars.shape[0]
    k_sat = c_vars.shape[1]
    degrees = np.zeros(n, dtype=np.int32)
    for i in range(m):
        for kk in range(k_sat):
            degrees[c_vars[i, kk]] += 1
    max_degree = 0
    for i in range(n):
        if degrees[i] > max_degree:
            max_degree = degrees[i]
    adj_indices = np.full((n, max_degree), -1, dtype=np.int32)
    adj_signs = np.full((n, max_degree), -1, dtype=np.int32)
    current_fill = np.zeros(n, dtype=np.int32)
    for i in range(m):
        for kk in range(k_sat):
            v = c_vars[i, kk]
            s = c_signs[i, kk]
            pos = current_fill[v]
            adj_indices[v, pos] = i
            adj_signs[v, pos] = s
            current_fill[v] += 1
    assignment = np.random.randint(0, 2, n)
    num_true_lits = np.zeros(m, dtype=np.int32)
    for i in range(m):
        count = 0
        for kk in range(k_sat):
            if assignment[c_vars[i, kk]] == c_signs[i, kk]:
                count += 1
        num_true_lits[i] = count
    unsat_buffer = np.empty(m, dtype=np.int32)
    for flip in range(1, max_flips + 1):
        unsat_count = 0
        for i in range(m):
            if num_true_lits[i] == 0:
                unsat_buffer[unsat_count] = i
                unsat_count += 1
        if unsat_count == 0:
            return flip
        target_c_idx = unsat_buffer[np.random.randint(0, unsat_count)]
        candidates = c_vars[target_c_idx]
        cand_breaks = np.zeros(k_sat, dtype=np.int32)
        cand_lmakes = np.zeros(k_sat, dtype=np.int32)
        has_zero_break = False
        for kk in range(k_sat):
            var = candidates[kk]
            current_break = 0
            make_1 = 0
            make_2 = 0
            deg = current_fill[var]
            for idx in range(deg):
                c_idx = adj_indices[var, idx]
                s = adj_signs[var, idx]
                lit_count = num_true_lits[c_idx]
                if assignment[var] == s:
                    if lit_count == 1:
                        current_break += 1
                else:
                    if lit_count == 0:
                        make_1 += 1
                    elif lit_count == 1:
                        make_2 += 1
            cand_breaks[kk] = current_break
            cand_lmakes[kk] = w1 * make_1 + w2 * make_2
            if current_break == 0:
                has_zero_break = True
        best_var = -1
        if has_zero_break:
            best_val = -1e9
            for kk in range(k_sat):
                if cand_breaks[kk] == 0:
                    score = cand_lmakes[kk]
                    if score > best_val:
                        best_val = score
                        best_var = candidates[kk]
                    elif score == best_val and np.random.random() < 0.5:
                        best_var = candidates[kk]
        elif np.random.random() < p_noise:
            best_var = candidates[np.random.randint(0, k_sat)]
        else:
            min_b = 999999
            max_l = -999999
            for kk in range(k_sat):
                b = cand_breaks[kk]
                lmk = cand_lmakes[kk]
                if b < min_b:
                    min_b = b
                    max_l = lmk
                    best_var = candidates[kk]
                elif b == min_b:
                    if lmk > max_l:
                        max_l = lmk
                        best_var = candidates[kk]
                    elif lmk == max_l and np.random.random() < 0.5:
                        best_var = candidates[kk]
        assignment[best_var] = 1 - assignment[best_var]
        deg = current_fill[best_var]
        for idx in range(deg):
            c_idx = adj_indices[best_var, idx]
            s = adj_signs[best_var, idx]
            if assignment[best_var] == s:
                num_true_lits[c_idx] += 1
            else:
                num_true_lits[c_idx] -= 1
    return max_flips

In [4]:
def clauses_to_numba_arrays(clauses, k: int):
    """BM24 (var, is_negated) -> Numba signs with literal true iff assignment == sign."""
    m = len(clauses)
    c_vars = np.zeros((m, k), dtype=np.int32)
    c_signs = np.zeros((m, k), dtype=np.int32)
    for i, clause in enumerate(clauses):
        for j, (var, is_negated) in enumerate(clause):
            c_vars[i, j] = int(var)
            c_signs[i, j] = 1 - int(bool(is_negated))
    return c_vars, c_signs


def generate_benchmark_dataset_cached(
    n_values,
    k: int,
    r: float,
    test_size: int,
    base_seed: int,
) -> Dict[int, List[dict]]:
    """SAT-filtered instances with cached H_diag and Numba clause arrays."""
    dataset: Dict[int, List[dict]] = {}
    for n in n_values:
        n = int(n)
        accepted = 0
        trial = 0
        instances = []
        pbar = tqdm(total=test_size, desc=f"dataset n={n}", leave=False)
        while accepted < test_size:
            if trial > 200_000:
                raise RuntimeError(f"too many rejections at n={n}")
            ss = np.random.SeedSequence([int(base_seed), n, accepted, trial])
            rng = np.random.default_rng(ss)
            lam = float(r) * n
            m_clauses = max(1, int(rng.poisson(lam)))
            clauses = [generate_random_clause(n, k, rng) for _ in range(m_clauses)]
            h_diag = build_h_diagonal(clauses, n)
            if not np.any(h_diag == 0):
                trial += 1
                continue
            c_vars, c_signs = clauses_to_numba_arrays(clauses, k)
            instances.append({
                "clauses": clauses,
                "h_diag": h_diag,
                "c_vars": c_vars,
                "c_signs": c_signs,
            })
            accepted += 1
            trial += 1
            pbar.update(1)
        pbar.close()
        dataset[n] = instances
    return dataset


def _seed_for_instance(base_seed: int, n: int, idx: int) -> int:
    return int(np.random.SeedSequence([base_seed, n, idx]).generate_state(1)[0])


def evaluate_classical_once(dataset, cfg) -> dict:
    """Run WalkSAT + WalkSATlm once; return median flips per n."""
    p_ws = float(cfg.get("walksat_p_noise", cfg.get("p_noise", 0.5)))
    p_lm = float(cfg.get("walksatlm_p_noise", 0.15))
    max_flips = int(cfg["max_flips"])
    w1, w2 = int(cfg["walksatlm_w1"]), int(cfg["walksatlm_w2"])
    base_seed = int(cfg["seed"])
    out = {"walksat": {}, "walksatlm": {}}
    for n in sorted(dataset.keys()):
        ws_flips, lm_flips = [], []
        for idx, inst in enumerate(tqdm(dataset[n], desc=f"classical n={n}", leave=False)):
            seed = _seed_for_instance(base_seed, n, idx)
            np.random.seed(seed)
            ws_flips.append(
                int(fast_walksat_solver(n, inst["c_vars"], inst["c_signs"], max_flips, p_ws))
            )
            np.random.seed(seed)
            lm_flips.append(
                int(walksatlm_paper_kernel(n, inst["c_vars"], inst["c_signs"], max_flips, p_lm, w1, w2))
            )
        out["walksat"][n] = float(np.median(ws_flips))
        out["walksatlm"][n] = float(np.median(lm_flips))
    return out


def fit_log2_slope(n_values, y_values) -> float:
    n_arr = np.asarray(n_values, dtype=float)
    y_arr = np.asarray(y_values, dtype=float)
    mask = np.isfinite(y_arr) & (y_arr > 0)
    if mask.sum() < 2:
        return float("nan")
    slope_nat = linregress(n_arr[mask], np.log(y_arr[mask])).slope
    return float(slope_nat / LN2)


def evaluate_lr_qaoa_depth(dataset, n_values, betas, gammas, eps: float = 1e-300) -> dict:
    """Median(1/p_succ) per n using cached H_diag (BM24 run_qaoa)."""
    med_rt = {}
    for n in n_values:
        costs = []
        for inst in dataset[int(n)]:
            psi = run_qaoa(inst["h_diag"], betas, gammas, int(n))
            p = per_instance_success_probability(psi, inst["h_diag"])
            costs.append(1.0 / max(float(p), eps))
        med_rt[int(n)] = float(np.median(costs))
    return med_rt

In [5]:
# --- One-time setup: benchmark data, classical baselines, training Hamiltonians ---
t0 = time.time()
n_values = list(range(int(CFG["n_min"]), int(CFG["n_max"]) + 1))

print("Building SAT benchmark dataset (cached H_diag)...")
dataset = generate_benchmark_dataset_cached(
    n_values, k=K, r=float(CFG["r"]), test_size=int(CFG["test_size"]), base_seed=int(CFG["seed"]),
)

print("Classical solvers (once)...")
classical = evaluate_classical_once(dataset, CFG)
ws_slope = fit_log2_slope(n_values, [classical["walksat"][n] for n in n_values])
lm_slope = fit_log2_slope(n_values, [classical["walksatlm"][n] for n in n_values])
print(f"  WalkSAT   (p_noise={CFG['walksat_p_noise']})   log2 slope = {ws_slope:.4f}")
print(f"  WalkSATlm (p_noise={CFG['walksatlm_p_noise']}) log2 slope = {lm_slope:.4f}")

print("Training Hamiltonians (once)...")
training_h = generate_training_h_diagonals(
    train_n=int(CFG["train_n"]),
    k=K,
    r=float(CFG["r"]),
    train_size=int(CFG["train_size"]),
    base_seed=int(CFG["seed"]),
    m_sampling="notebook",
)
print(f"Setup done in {time.time() - t0:.1f}s")

Building SAT benchmark dataset (cached H_diag)...


dataset n=12:   0%|          | 0/200 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# --- Depth sweep: train + LR benchmark only (always runs ALL depths in CFG["depths"]) ---
from bm24_run_io import make_run_stem

trace = []
run_stem = make_run_stem("efficient-scaling")
print(f"Run id: {run_stem}")
depths_to_run = [int(p) for p in CFG["depths"]]
print(f"Will run {len(depths_to_run)} depths: {depths_to_run[0]} … {depths_to_run[-1]} (no early stop on win)")

for depth in depths_to_run:
    depth = int(depth)
    t_depth = time.time()
    print(f"\n{'=' * 60}\nDepth p = {depth}\n{'=' * 60}")

    _, diag = train_lr_grid_search_bm24(
        training_h,
        train_n=int(CFG["train_n"]),
        depth=depth,
        skip_grid=bool(CFG["skip_grid"]),
        beta_schedule=str(CFG["lr_beta_schedule"]),
        cobyla_maxiter=int(CFG["cobyla_maxiter"]),
    )
    dg, db = diag["best_deltas"]
    betas, gammas = make_lr_angles(
        float(dg), float(db), depth,
        beta_schedule=str(CFG["lr_beta_schedule"]),
        angle_convention="bm24",
    )
    print(f"  trained dg={dg:.6f} db={db:.6f} train_p={diag['best_avg_train_p_succ']:.4e}")

    med_rt = evaluate_lr_qaoa_depth(dataset, n_values, betas, gammas)
    lr_slope = fit_log2_slope(n_values, [med_rt[n] for n in n_values])
    beats_ws = bool(np.isfinite(lr_slope) and np.isfinite(ws_slope) and lr_slope < ws_slope)
    beats_lm = bool(np.isfinite(lr_slope) and np.isfinite(lm_slope) and lr_slope < lm_slope)

    row = {
        "depth": depth,
        "delta_gamma": float(dg),
        "delta_beta": float(db),
        "lr_log2_slope": lr_slope,
        "walksat_log2_slope": ws_slope,
        "walksatlm_log2_slope": lm_slope,
        "median_runtime_per_n": {str(n): med_rt[n] for n in n_values},
        "lr_beats_walksat_scaling": beats_ws,
        "lr_beats_walksatlm_scaling": beats_lm,
        "elapsed_s": time.time() - t_depth,
    }
    trace.append(row)
    print(f"  LR log2 slope={lr_slope:.4f}  (WS={ws_slope:.4f} LM={lm_slope:.4f})")
    print(f"  beats WS={beats_ws} beats LM={beats_lm}  elapsed={row['elapsed_s']:.1f}s")

out_json = CFG["output_dir"] / f"{run_stem}.json"
with open(out_json, "w", encoding="utf-8") as f:
    json.dump({"config": {k: (list(v) if isinstance(v, range) else v) for k, v in CFG.items()}, "trace": trace}, f, indent=2, default=str)
print(f"\nWrote {out_json}")

In [ ]:
# --- Summary plot (same y as pipeline scaling-vs-depth) ---
depths = [row["depth"] for row in trace]
lr_slopes = [row["lr_log2_slope"] for row in trace]
win_depths = [row["depth"] for row in trace if row["lr_beats_walksat_scaling"] and row["lr_beats_walksatlm_scaling"]]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(depths, lr_slopes, "o-", color="C0", linewidth=2, markersize=7, label="LR-QAOA (median 1/p)")
ax.axhline(ws_slope, color="C1", linestyle="--", linewidth=1.5, label=f"WalkSAT ({ws_slope:.3f})")
ax.axhline(lm_slope, color="C2", linestyle="--", linewidth=1.5, label=f"WalkSATlm ({lm_slope:.3f})")
if CFG.get("annotate_first_win", False) and win_depths:
    p0 = min(win_depths)
    ax.axvline(p0, color="0.4", linestyle=":", alpha=0.8)
    ax.scatter([p0], [lr_slopes[depths.index(p0)]], s=120, facecolors="none", edgecolors="C0", linewidths=2)
    ax.annotate(f"beats both at p={p0}", xy=(p0, lr_slopes[depths.index(p0)]), xytext=(8, 12), textcoords="offset points", fontsize=9)
elif win_depths:
    print(f"(info) first depth beating both classical slopes: p={min(win_depths)} (annotate_first_win=False)")
ax.set_xlabel("QAOA depth p")
ax.set_ylabel(r"$\log_2$ slope of median cost vs $n$")
from bm24_run_io import format_benchmark_title

ax.set_title(
    format_benchmark_title(
        {**CFG, "k": K},
        headline="Efficient LR sweep",
        depths=depths,
    ),
    fontsize=10,
)
ax.legend(loc="best", fontsize=9)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plot_path = CFG["output_dir"] / f"{run_stem}.png"
fig.savefig(plot_path, dpi=150)
plt.show()
print(f"Saved {plot_path}")